## Load All 4 Feature Files

In [ ]:
import pandas as pd
import numpy as np

df_billings = pd.read_csv('../../../data/interim/billings_features.csv')
df_renewal  = pd.read_csv('../../../data/interim/renewal_calls_features.csv')
df_email = pd.read_csv('../../../data/interim/email_features.csv')
df_cc = pd.read_csv('../../../data/interim/cc_calls_features.csv')

print(f"Billings : {df_billings.shape}")
print(f"Renewal Calls : {df_renewal.shape}")
print(f"Emails : {df_email.shape}")
print(f"CC Calls : {df_cc.shape}")

Billings      : (113769, 63)
Renewal Calls : (17458, 39)
Emails        : (21362, 33)
CC Calls      : (17548, 46)


## Verify co_ref + renewal_year is in All Files

In [19]:
for name, df in [('billings', df_billings), ('renewal', df_renewal),
                 ('email', df_email), ('cc', df_cc)]:
    has_coref = 'co_ref' in df.columns
    has_year  = 'renewal_year' in df.columns
    unique    = df['co_ref'].nunique() if has_coref else 0
    dupes     = df.duplicated(subset=['co_ref', 'renewal_year']).sum() if (has_coref and has_year) else 'N/A'
    print(f"{name:12s} — co_ref: {has_coref} | renewal_year: {has_year} | unique co_ref: {unique:,} | duplicate keys: {dupes}")

billings     — co_ref: True | renewal_year: True | unique co_ref: 46,328 | duplicate keys: 0
renewal      — co_ref: True | renewal_year: True | unique co_ref: 14,655 | duplicate keys: 0
email        — co_ref: True | renewal_year: True | unique co_ref: 21,362 | duplicate keys: 0
cc           — co_ref: True | renewal_year: True | unique co_ref: 14,988 | duplicate keys: 0


## Verify Billings Composite Key is Unique

Billings must have zero duplicate co_ref + renewal_year combinations before joining.

In [20]:
dupes = df_billings.duplicated(subset=['co_ref', 'renewal_year']).sum()
print(f"Duplicate co_ref + renewal_year in billings: {dupes}")
print(f"Total billings rows                        : {len(df_billings):,}")
# dupes should be 0

Duplicate co_ref + renewal_year in billings: 0
Total billings rows                        : 113,769


## Join All Datasets onto Billings

joining on co_ref + renewal_year instead of co_ref alone. This ensures calls and emails from year 2024 only attach to the 2024 renewal record — not all renewal records for that customer.

In [21]:
df = df_billings.copy()

# KEY FIX: join on ['co_ref', 'renewal_year'] not just 'co_ref'
df = df.merge(df_renewal, on=['co_ref', 'renewal_year'], how='left')
df = df.merge(df_email,   on=['co_ref', 'renewal_year'], how='left')
df = df.merge(df_cc,      on=['co_ref', 'renewal_year'], how='left')

print(f"Shape after join : {df.shape}")

Shape after join : (113769, 175)


In [22]:
df.rename(columns={'high_risk_call_y': 'high_risk_call'}, inplace=True)
df.rename(columns={'dissatisfaction_issue_count_y': 'dissatisfaction_issue_count'}, inplace=True)

## Fill NaN from Unmatched Rows

Customers with no calls/emails in a given renewal year get NaN — fill with 0 since no interaction = 0.

In [23]:
renewal_cols = [c for c in df_renewal.columns if c not in ['co_ref', 'renewal_year']]
email_cols = [c for c in df_email.columns   if c not in ['co_ref', 'renewal_year']]
cc_cols = [c for c in df_cc.columns      if c not in ['co_ref', 'renewal_year']]

df[renewal_cols] = df[renewal_cols].fillna(0)
df[email_cols] = df[email_cols].fillna(0)
df[cc_cols] = df[cc_cols].fillna(0)



## Add Interaction Flags

In [24]:
df['has_renewal_call'] = (df['rc_14d_total_calls'] > 0).astype(int)
df['has_email'] = (df['email_count'] > 0).astype(int)
df['has_cc_call'] = (df['cc_total_calls'] > 0).astype(int)

print(f"Customers with renewal call : {df['has_renewal_call'].sum():,} ({df['has_renewal_call'].mean()*100:.1f}%)")
print(f"Customers with email : {df['has_email'].sum():,} ({df['has_email'].mean()*100:.1f}%)")
print(f"Customers with CC call : {df['has_cc_call'].sum():,} ({df['has_cc_call'].mean()*100:.1f}%)")

Customers with renewal call : 17,458 (15.3%)
Customers with email : 20,214 (17.8%)
Customers with CC call : 16,192 (14.2%)


C:\Users\gopik\AppData\Local\Temp\ipykernel_23592\1114759762.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_renewal_call'] = (df['rc_14d_total_calls'] > 0).astype(int)
C:\Users\gopik\AppData\Local\Temp\ipykernel_23592\1114759762.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_email'] = (df['email_count'] > 0).astype(int)
C:\Users\gopik\AppData\Local\Temp\ipykernel_23592\1114759762.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

## Drop ID and Non-Feature Columns

In [25]:
drop_cols = [
    'co_ref', # ID — not a feature
    'renewal_month', # raw date — not needed after features are created
]

df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f"Final shape: {df.shape}")

Final shape: (113769, 176)


## Save Final Dataset

In [26]:
df.to_csv('../../../data/processed/final_dataset.csv', index=False)